# 03 - Train the acoustic model

About 7 to 9 hours on a 4090, roughly 5 to 7 dollars on RunPod.

Every `sample_every` steps the probe sentences are rendered to TensorBoard, so
you can *listen* to whether homographs are pronounced correctly instead of
guessing from a loss curve.

In [2]:
import os, sys
REPO = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
os.environ["PYTHONIOENCODING"] = "utf-8"

# ---------------------------------------------------------------------------
# PICK YOUR EXPERIMENT HERE. This is the only line to change.
#
#   configs/exp0_small.yaml     2013 clips, ~2 GB   -> proves the pipeline,
#                                                     runs on a 6 GB GPU
#   configs/exp1_egyptian.yaml  15.6k clips, 68 h   -> the real run
# ---------------------------------------------------------------------------
CONFIG = "configs/exp1_egyptian.yaml"

from adaptts.utils.config import load_config
from adaptts.utils.logging_utils import setup_logging
setup_logging()
cfg = load_config(CONFIG)
print("repo   :", REPO)
print("config :", CONFIG, "->", cfg.name)
print("dataset:", cfg.paths.hf_dataset_id)

# The homographs named in the brief, read from the probe file so the notebooks
# never hardcode a word list of their own.
import json as _json
PROBE_WORDS = sorted({
    w for _s in _json.load(open("assets/probe_sentences.json", encoding="utf-8"))["sentences"]
    for w in _s["focus"].split(" / ") if w and w != "none"
})
print("probe  :", " ".join(PROBE_WORDS))

repo   : /workspace/AdapTTS
config : configs/exp1_egyptian.yaml -> exp1_egyptian_homograph
dataset: ehabnegm/100-hour-Egyptian-dataset-single-speaker
probe  : الدول دول علم مصر


## Check the model size before committing GPU hours

In [3]:
import torch
from adaptts.models.acoustic import AcousticModel
from adaptts.text.vocab import CharVocab

vocab = CharVocab.load(cfg.paths.charvocab_path)
m = AcousticModel(
    len(vocab), n_quantizers=cfg.audio.n_quantizers, codebook_size=cfg.audio.codebook_size,
    d_model=cfg.acoustic.d_model, n_layers=cfg.acoustic.n_layers, n_heads=cfg.acoustic.n_heads,
    d_ff=cfg.acoustic.d_ff, text_d_model=cfg.acoustic.text_d_model,
    text_n_layers=cfg.acoustic.text_n_layers, text_n_heads=cfg.acoustic.text_n_heads,
    depth_d_model=cfg.acoustic.depth_d_model, depth_n_layers=cfg.acoustic.depth_n_layers,
    depth_n_heads=cfg.acoustic.depth_n_heads, speaker_dim=cfg.acoustic.speaker_dim,
    max_codes=cfg.discovery.max_codes_per_word, pc_embed_dim=cfg.acoustic.pc_embed_dim,
    exit_layers=cfg.acoustic.exit_layers, pad_id=vocab.pad_id,
)
n = sum(p.numel() for p in m.parameters())
print(f"acoustic model: {n / 1e6:.1f}M parameters")
print(f"fp32 weights:   {n * 4 / 1024 ** 2:.0f} MB")
print(f"total deployed with the frozen Mimi decoder: about {(n + 25e6) / 1e6:.0f}M")
del m

acoustic model: 56.0M parameters
fp32 weights:   214 MB
total deployed with the frozen Mimi decoder: about 81M


## TensorBoard

Watch `train/acc_q0` for the coarse RVQ level, and the `probe/` audio tab.

In [4]:
%load_ext tensorboard
%tensorboard --logdir $cfg.paths.tb_dir --port 6006 --bind_all

Reusing TensorBoard on port 6006 (pid 5453), started 0:28:00 ago. (Use '!kill 5453' to kill it.)

## Train

If the pod restarts, resume with
`--resume runs/exp1/checkpoints/acoustic/last.pt`.

In [19]:
!python scripts/train_acoustic.py --config $CONFIG --resume runs/exp1/checkpoints/acoustic/last.pt

02:30:41 info  train_acoustic AdapTTS stage C: adaptive-depth acoustic model
02:30:41 info  train_acoustic device: cuda
02:30:41 info  train_acoustic configuration
02:30:41 info  train_acoustic setting     value                                     
02:30:41 info  train_acoustic ------------------------------------------------------
02:30:41 info  train_acoustic experiment  exp1_egyptian_homograph                   
02:30:41 info  train_acoustic dataset     /workspace/AdapTTS/data/masri100h         
02:30:41 info  train_acoustic cache       /workspace/AdapTTS/cache/exp1             
02:30:41 info  train_acoustic run dir     /workspace/AdapTTS/runs/exp1              
02:30:41 info  train_acoustic codec       kyutai/mimi (8 quantizers)                
02:30:41 info  train_acoustic teacher     aubmindlab/bert-base-arabertv02-twitter   
02:30:41 info  train_acoustic aligner     MahmoudAshraf/mms-300m-1130-forced-aligner
02:30:41 info  train_acoustic precision   bf16                         

## Quick machinery check

Before listening critically, confirm the parts are working: real samples, a
measurable difference between exit depths, and an override that actually
changes the output. Audio quality at this point depends entirely on how long
the model trained.

In [8]:
!python scripts/smoke_generate.py --config $CONFIG --device cpu

01:29:35 info  httpx          HTTP Request: HEAD https://huggingface.co/kyutai/mimi/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
01:29:36 info  httpx          HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/kyutai/mimi/89091b3e466eb6a9d11e537bf26b144f194978f7/config.json?%2Fkyutai%2Fmimi%2Fresolve%2Fmain%2Fconfig.json=&etag=%22907c57dc64c983785bfc68572e4e8ab8eddd64b8%22 "HTTP/1.1 200 OK"
01:29:36 info  httpx          HTTP Request: HEAD https://huggingface.co/kyutai/mimi/resolve/main/model.safetensors "HTTP/1.1 302 Found"
Loading weights: 100%|██████████████████████| 350/350 [00:00<00:00, 827.00it/s]

LOADED
01:29:37 info  smoke          component         status
01:29:37 info  smoke          ------------------------
01:29:37 info  smoke          context encoder   yes   
01:29:37 info  smoke          acoustic model    yes   
01:29:37 info  smoke          codec             yes   
01:29:37 info  smoke          discovered words  145   
01:29:37 info  smoke  

## Listen to the probe set

In [22]:
import json
from IPython.display import Audio, display
from adaptts.infer.pipeline import AdapTTS

tts = AdapTTS.from_checkpoints(CONFIG, device="cpu")
probes = json.load(open("assets/probe_sentences.json", encoding="utf-8"))["sentences"]

for p in probes:
    plan = tts.analyze(p["text"])
    wav, st = tts.synthesize(plan)
    print()
    print(f"=== {p['tag']} | expected {p['expected']} ===")
    print("   ", p["text"])
    print(f"    depth {st['depth']}  difficulty {st['sentence_difficulty']:.2f}  "
          f"RTF {st['real_time_factor']:.2f}")
    display(Audio(wav, rate=st["sample_rate"]))

08:11:31 info  httpx          HTTP Request: HEAD https://huggingface.co/kyutai/mimi/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
08:11:31 info  httpx          HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/kyutai/mimi/89091b3e466eb6a9d11e537bf26b144f194978f7/config.json?%2Fkyutai%2Fmimi%2Fresolve%2Fmain%2Fconfig.json=&etag=%22907c57dc64c983785bfc68572e4e8ab8eddd64b8%22 "HTTP/1.1 200 OK"
08:11:31 info  httpx          HTTP Request: HEAD https://huggingface.co/kyutai/mimi/resolve/main/model.safetensors "HTTP/1.1 302 Found"


Loading weights:   0%|          | 0/350 [00:00<?, ?it/s]


=== 3alam_flag | expected عَلَم ===
    انا شوفت علم مصر بيرفرف
    depth 4  difficulty 0.00  RTF 1.19



=== 3ilm_science | expected عِلْم ===
    علم الفيزيا من اهم العلوم البشرية
    depth 4  difficulty 0.00  RTF 0.78



=== masr_egypt | expected مَصْر ===
    مصر عندها امكانيات و موارد كتير جدا
    depth 4  difficulty 0.00  RTF 1.05



=== musirr_insisting | expected مُصِرّ ===
    انا كنت مصر على الراي بتاعي لحد النهاية
    depth 4  difficulty 0.00  RTF 1.00



=== dowal_countries | expected الدِوَل ===
    الدول دي بتتنافس على السوق العالمي
    depth 4  difficulty 0.00  RTF 1.09



=== dool_these | expected دُول ===
    الناس دول شايفين نفسهم احسن من غيرهم
    depth 4  difficulty 0.00  RTF 0.83



=== multi_homograph | expected مُصِرّ ... مَصْر ... الدِوَل ... دُول ===
    انا كنت مصر على ان مصر عندها امكانيات تخليها تتفوق على دول من اللي شايفين نفسهم دول
    depth 4  difficulty 0.00  RTF 0.82



=== easy_control | expected no ambiguity ===
    الجو النهارده حلو جدا و الشمس طالعة
    depth 4  difficulty 0.00  RTF 1.07


## The controllability demo

The same sentence rendered with each reading, by overriding the code. No
diacritics are typed anywhere.

In [10]:
text = "انا شوفت علم مصر بيرفرف"
plan = tts.analyze(text)
print(plan)

w = [x for x in plan.hard_words if x.word == "علم"][0]
for c in range(w.n_codes):
    plan.set_code("علم", c)
    wav, st = tts.synthesize(plan)
    print()
    print(f"--- علم forced to code {c} ---")
    display(Audio(wav, rate=st["sample_rate"]))

text: انا شوفت علم مصر بيرفرف
sentence difficulty: 0.000   depth: 4
no ambiguous words: every word has a single known reading


IndexError: list index out of range

## Adaptive depth: measure the saving

An easy sentence should route to a shallow exit and be measurably faster.

In [21]:
import time

easy = "الجو النهارده حلو جدا و الشمس طالعة"
hard = "انا كنت مصر على ان مصر عندها امكانيات تخليها تتفوق على دول"

for name, txt in [("easy", easy), ("hard", hard)]:
    plan = tts.analyze(txt)
    t0 = time.perf_counter()
    wav, st = tts.synthesize(plan)
    dt = time.perf_counter() - t0
    print(f"{name:5s} difficulty {plan.sentence_difficulty:.3f} -> depth {st['depth']:2d}  "
          f"{dt:.2f}s for {st['audio_seconds']:.1f}s audio  RTF {st['real_time_factor']:.2f}")

easy  difficulty 0.000 -> depth  4  2.22s for 6.6s audio  RTF 0.33
hard  difficulty 0.000 -> depth  4  3.12s for 9.0s audio  RTF 0.34


In [12]:
# Force each depth on the same sentence to isolate the compute saving.
plan = tts.analyze(hard)
for d in cfg.acoustic.exit_layers:
    plan.set_depth(d)
    t0 = time.perf_counter()
    wav, st = tts.synthesize(plan)
    dt = time.perf_counter() - t0
    print(f"depth {d:2d}: {dt:.2f}s  RTF {st['real_time_factor']:.2f}")
    display(Audio(wav, rate=st["sample_rate"]))

depth  4: 7.42s  RTF 0.91


depth  8: 8.12s  RTF 1.00


depth 12: 9.30s  RTF 1.14
